In [17]:
"""
full_logistic_dirac_pipeline.py

Full logistic-Dirac CTMC on V ∪ E.

State space:
    V = {0,1,...,K}
    E = {e_0,...,e_{K-1}}, where e_i connects i and i+1

Total dimension:
    n = |V| + |E| = (K+1) + K = 2K+1

Generator model:
    The CTMC lives on the full space V ∪ E.

    Vertex -> edge transitions:
        i -> e_i     at rate lambda_i   (birth channel), if i < K
        i -> e_{i-1} at rate mu_i       (death channel), if i > 0

    Edge -> vertex transitions:
        e_i -> i     at rate kappa_left
        e_i -> i+1   at rate kappa_right

    where
        lambda_i = a + r*i*(1 - i/K), for i < K
        mu_i     = d*i, for i > 0

This gives a full generator Q on V ∪ E.

The script:
1. builds the graph and Dirac operator on V ∪ E
2. builds the true full generator Q*
3. generates endpoint-only data D = {(x_k,y_k,t_k)}
4. fits the parameter vector (r,d,a,kappa_left,kappa_right)
5. computes Dirac coefficients and reconstruction
6. visualizes everything
7. increases sample size automatically until relative error < 1e-2, or a max budget is reached

This is aligned with your full-space generator + Dirac decomposition setup. 
"""

from __future__ import annotations

import json
import math
import textwrap
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from scipy.linalg import expm, eigh
from scipy.optimize import minimize


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Config:
    # Graph / model size
    K: int = 10

    # True parameters
    r_true: float = 0.9
    d_true: float = 0.25
    a_true: float = 0.10
    kappa_left_true: float = 2.0
    kappa_right_true: float = 2.0

    # Data generation
    N_obs_initial: int = 12000
    N_obs_multiplier: float = 1.75
    N_obs_max: int = 120000
    fixed_time: float = 1.0
    random_seed: int = 123

    # Start distribution mode on full V ∪ E
    start_distribution_mode: str = "stationary"   # "stationary" or "uniform"

    # Optimization
    n_restarts: int = 8
    maxiter: int = 400
    bounds_r: Tuple[float, float] = (1e-6, 3.0)
    bounds_d: Tuple[float, float] = (1e-6, 3.0)
    bounds_a: Tuple[float, float] = (0.0, 2.0)
    bounds_kappa_left: Tuple[float, float] = (1e-4, 8.0)
    bounds_kappa_right: Tuple[float, float] = (1e-4, 8.0)

    # Target
    target_relative_error: float = 1e-2

    # Output
    outdir: str = "full_logistic_dirac_outputs"


# =============================================================================
# Utilities
# =============================================================================

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def savefig(fig: plt.Figure, path: Path, dpi: int = 180) -> None:
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


def stable_row_normalize(P: NDArray[np.float64]) -> NDArray[np.float64]:
    P = np.maximum(P, 0.0)
    rs = P.sum(axis=1, keepdims=True)
    rs = np.where(rs <= 1e-15, 1.0, rs)
    return P / rs


def sample_from_probs(rng: np.random.Generator, probs: NDArray[np.float64]) -> int:
    probs = np.maximum(probs, 0.0)
    s = probs.sum()
    if s <= 1e-15:
        probs = np.ones_like(probs) / len(probs)
    else:
        probs = probs / s
    return int(rng.choice(len(probs), p=probs))


def hs_norm_sq(A: NDArray[np.float64]) -> float:
    return float(np.sum(A * A))


# =============================================================================
# Graph and Dirac operator on V ∪ E
# =============================================================================

def build_path_graph(K: int) -> Dict[str, object]:
    vertices = list(range(K + 1))
    edges = [(i, i + 1) for i in range(K)]

    A_v = np.zeros((K + 1, K + 1), dtype=float)
    for i, j in edges:
        A_v[i, j] = 1.0
        A_v[j, i] = 1.0

    return {"vertices": vertices, "edges": edges, "adjacency_vertices": A_v}


def build_incidence_matrix(K: int) -> NDArray[np.float64]:
    B = np.zeros((K, K + 1), dtype=float)
    for i in range(K):
        B[i, i] = -1.0
        B[i, i + 1] = 1.0
    return B


def build_dirac_operator(K: int) -> NDArray[np.float64]:
    """
    Dirac operator on l^2(V) ⊕ l^2(E):
        D = [ 0   B^T ]
            [ B    0  ]
    with vertices ordered first, then edges.
    """
    B = build_incidence_matrix(K)
    Nv = K + 1
    Ne = K
    D = np.block([
        [np.zeros((Nv, Nv), dtype=float), B.T],
        [B, np.zeros((Ne, Ne), dtype=float)]
    ])
    return D


# =============================================================================
# Full logistic generator on V ∪ E
# =============================================================================

def build_full_logistic_vertex_edge_generator(
    K: int,
    r: float,
    d: float,
    a: float,
    kappa_left: float,
    kappa_right: float
) -> NDArray[np.float64]:
    """
    Full generator on V ∪ E.

    State ordering:
        [0,1,...,K, e_0,e_1,...,e_{K-1}]
    """
    Nv = K + 1
    Ne = K
    n = Nv + Ne

    Q = np.zeros((n, n), dtype=float)

    def v_idx(i: int) -> int:
        return i

    def e_idx(i: int) -> int:
        return Nv + i

    # Vertex -> edge (logistic channels)
    for i in range(Nv):
        lam = (a + r * i * (1.0 - i / K)) if i < K else 0.0
        mu = d * i if i > 0 else 0.0

        if i < K:
            Q[v_idx(i), e_idx(i)] = max(lam, 0.0)
        if i > 0:
            Q[v_idx(i), e_idx(i - 1)] = max(mu, 0.0)

    # Edge -> vertex
    for i in range(Ne):
        Q[e_idx(i), v_idx(i)] = max(kappa_left, 0.0)
        Q[e_idx(i), v_idx(i + 1)] = max(kappa_right, 0.0)

    # Diagonal = negative row sums
    np.fill_diagonal(Q, -Q.sum(axis=1))
    return Q


def stationary_distribution(Q: NDArray[np.float64]) -> NDArray[np.float64]:
    vals, vecs = np.linalg.eig(Q.T)
    idx = np.argmin(np.abs(vals))
    v = np.real(vecs[:, idx])
    v = np.abs(v)
    s = v.sum()
    if s <= 1e-15:
        return np.ones(Q.shape[0], dtype=float) / Q.shape[0]
    return v / s


# =============================================================================
# Synthetic endpoint data on full state space
# =============================================================================

def generate_endpoint_dataset_fixed_time(
    Q_true: NDArray[np.float64],
    N_obs: int,
    T: float,
    rho0: NDArray[np.float64],
    seed: int
) -> NDArray[np.float64]:
    rng = np.random.default_rng(seed)
    data = np.zeros((N_obs, 3), dtype=float)

    P = stable_row_normalize(expm(Q_true * T))

    for m in range(N_obs):
        x = sample_from_probs(rng, rho0)
        y = sample_from_probs(rng, P[x])
        data[m] = [x, y, T]

    return data


def empirical_transition_counts(data: NDArray[np.float64], n: int) -> NDArray[np.float64]:
    counts = np.zeros((n, n), dtype=float)
    for x, y, _t in data:
        counts[int(x), int(y)] += 1
    return counts


# =============================================================================
# Dirac decomposition
# =============================================================================

def dirac_spectral_projectors(
    D: NDArray[np.float64]
) -> Tuple[NDArray[np.float64], NDArray[np.float64], List[NDArray[np.float64]]]:
    eigvals, eigvecs = eigh(D)
    projectors: List[NDArray[np.float64]] = []
    for j in range(D.shape[0]):
        u = eigvecs[:, j:j+1]
        projectors.append(u @ u.T)
    return eigvals, eigvecs, projectors


def dirac_coefficients(
    Q: NDArray[np.float64],
    D: NDArray[np.float64],
    projectors: List[NDArray[np.float64]]
) -> Tuple[NDArray[np.float64], NDArray[np.float64]]:
    n = Q.shape[0]
    a = np.zeros(n, dtype=float)
    b = np.zeros((n, n), dtype=float)

    for mu in range(n):
        Mmu = projectors[mu]
        a[mu] = float(np.trace(Mmu.T @ Q))

    for mu in range(n):
        Mmu = projectors[mu]
        for nu in range(n):
            if mu == nu:
                continue
            Mnu = projectors[nu]
            b[mu, nu] = float(np.trace(Mmu @ Q @ Mnu @ D))

    return a, b


def reconstruct_from_dirac_coefficients(
    a: NDArray[np.float64],
    b: NDArray[np.float64],
    D: NDArray[np.float64],
    projectors: List[NDArray[np.float64]]
) -> NDArray[np.float64]:
    n = len(projectors)
    Q_rec = np.zeros((n, n), dtype=float)

    for mu in range(n):
        Q_rec += a[mu] * projectors[mu]

    for mu in range(n):
        for nu in range(n):
            if mu == nu:
                continue
            Q_rec += b[mu, nu] * (projectors[nu] @ D @ projectors[mu].T)

    return Q_rec


# =============================================================================
# Likelihood and fitting
# =============================================================================

def endpoint_loglik_full_fixed_time(Q: NDArray[np.float64], data: NDArray[np.float64]) -> float:
    T = float(data[0, 2])
    P = stable_row_normalize(expm(Q * T))

    x_idx = data[:, 0].astype(int)
    y_idx = data[:, 1].astype(int)
    probs = np.maximum(P[x_idx, y_idx], 1e-14)
    return float(np.sum(np.log(probs)))


def objective_from_params(
    theta: NDArray[np.float64],
    K: int,
    data: NDArray[np.float64],
) -> float:
    r, d, a, kappa_left, kappa_right = map(float, theta)
    Q = build_full_logistic_vertex_edge_generator(
        K, r, d, a, kappa_left, kappa_right
    )
    return -endpoint_loglik_full_fixed_time(Q, data)


def run_multistart_fit(
    cfg: Config,
    data: NDArray[np.float64]
) -> Tuple[NDArray[np.float64], object]:
    rng = np.random.default_rng(cfg.random_seed + 999)
    starts: List[NDArray[np.float64]] = []

    # Truth-adjacent starts
    starts.append(np.array([
        0.90 * cfg.r_true,
        0.90 * cfg.d_true,
        0.90 * cfg.a_true,
        0.90 * cfg.kappa_left_true,
        0.90 * cfg.kappa_right_true
    ], dtype=float))

    starts.append(np.array([
        1.10 * cfg.r_true,
        1.10 * cfg.d_true,
        1.10 * cfg.a_true,
        1.10 * cfg.kappa_left_true,
        1.10 * cfg.kappa_right_true
    ], dtype=float))

    starts.append(np.array([
        cfg.r_true,
        cfg.d_true,
        cfg.a_true,
        cfg.kappa_left_true,
        cfg.kappa_right_true
    ], dtype=float))

    while len(starts) < cfg.n_restarts:
        starts.append(np.array([
            rng.uniform(*cfg.bounds_r),
            rng.uniform(*cfg.bounds_d),
            rng.uniform(*cfg.bounds_a),
            rng.uniform(*cfg.bounds_kappa_left),
            rng.uniform(*cfg.bounds_kappa_right),
        ], dtype=float))

    bounds = [
        cfg.bounds_r,
        cfg.bounds_d,
        cfg.bounds_a,
        cfg.bounds_kappa_left,
        cfg.bounds_kappa_right,
    ]

    best_res = None
    best_val = np.inf

    for x0 in starts:
        res = minimize(
            objective_from_params,
            x0=x0,
            args=(cfg.K, data),
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": cfg.maxiter}
        )
        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    if best_res is None:
        raise RuntimeError("All optimization restarts failed.")
    return best_res.x, best_res


# =============================================================================
# Visualization
# =============================================================================

def plot_graph(vertices: List[int], edges: List[Tuple[int, int]], outpath: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 2.8))
    x = np.array(vertices, dtype=float)
    y = np.zeros_like(x)

    ax.plot(x, y, 'o', ms=9)
    for i, j in edges:
        ax.plot([i, j], [0, 0], '-', lw=2.0)

    for v in vertices:
        ax.text(v, 0.05, str(v), ha='center', va='bottom', fontsize=10)

    ax.set_title("Path graph on vertices")
    ax.set_xlabel("vertex state")
    ax.set_yticks([])
    savefig(fig, outpath)


def plot_matrix(A: NDArray[np.float64], title: str, outpath: Path, sep: int | None = None) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(A, aspect='auto')
    ax.set_title(title)
    ax.set_xlabel("column")
    ax.set_ylabel("row")
    if sep is not None:
        ax.axhline(sep - 0.5, color='w', lw=1.5)
        ax.axvline(sep - 0.5, color='w', lw=1.5)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    savefig(fig, outpath)


def plot_dirac_spectrum(eigvals: NDArray[np.float64], outpath: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.stem(np.arange(1, len(eigvals) + 1), eigvals)
    ax.set_title("Spectrum of the Dirac operator")
    ax.set_xlabel("index")
    ax.set_ylabel("eigenvalue")
    ax.grid(True, alpha=0.3)
    savefig(fig, outpath)


def plot_dirac_eigenvectors(eigvecs: NDArray[np.float64], n_show: int, outpath: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 4))
    xs = np.arange(eigvecs.shape[0])
    for j in range(min(n_show, eigvecs.shape[1])):
        ax.plot(xs, eigvecs[:, j], lw=1.6, label=f"u_{j+1}")
    ax.set_title("First Dirac eigenvectors on V ∪ E")
    ax.set_xlabel("basis index")
    ax.set_ylabel("component")
    ax.legend()
    ax.grid(True, alpha=0.3)
    savefig(fig, outpath)


def plot_counts_matrix(counts: NDArray[np.float64], outpath: Path, sep: int) -> None:
    plot_matrix(counts, "Empirical endpoint counts on V ∪ E", outpath, sep=sep)


def plot_true_vs_fit(Q_true: NDArray[np.float64], Q_hat: NDArray[np.float64], outpath: Path, sep: int) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5))
    mats = [Q_true, Q_hat, Q_hat - Q_true]
    titles = ["True full generator $Q^*$", "Fitted full generator $\\widehat Q$", "$\\widehat Q - Q^*$"]

    for ax, A, title in zip(axes, mats, titles):
        im = ax.imshow(A, aspect='auto')
        ax.set_title(title)
        ax.set_xlabel("column")
        ax.set_ylabel("row")
        ax.axhline(sep - 0.5, color='w', lw=1.2)
        ax.axvline(sep - 0.5, color='w', lw=1.2)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    savefig(fig, outpath)


def plot_transition_kernels(Q_true: NDArray[np.float64], Q_hat: NDArray[np.float64], times: List[float], outdir: Path, sep: int) -> None:
    for t in times:
        P_true = stable_row_normalize(expm(Q_true * t))
        P_hat = stable_row_normalize(expm(Q_hat * t))

        fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5))
        mats = [P_true, P_hat, P_hat - P_true]
        titles = [f"$e^{{tQ^*}}$ at t={t:.2f}", f"$e^{{t\\widehat Q}}$ at t={t:.2f}", f"Difference at t={t:.2f}"]

        for ax, A, title in zip(axes, mats, titles):
            im = ax.imshow(A, aspect='auto')
            ax.set_title(title)
            ax.set_xlabel("end state")
            ax.set_ylabel("start state")
            ax.axhline(sep - 0.5, color='w', lw=1.2)
            ax.axvline(sep - 0.5, color='w', lw=1.2)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        savefig(fig, outdir / f"transition_kernel_t_{str(t).replace('.', '_')}.png")


def plot_decomposition(Q_true: NDArray[np.float64], Q_rec: NDArray[np.float64], outpath: Path, sep: int) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5))
    mats = [Q_true, Q_rec, Q_rec - Q_true]
    titles = ["Original full generator", "Dirac reconstruction", "Reconstruction error"]

    for ax, A, title in zip(axes, mats, titles):
        im = ax.imshow(A, aspect='auto')
        ax.set_title(title)
        ax.set_xlabel("column")
        ax.set_ylabel("row")
        ax.axhline(sep - 0.5, color='w', lw=1.2)
        ax.axvline(sep - 0.5, color='w', lw=1.2)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    savefig(fig, outpath)


def plot_parameter_comparison(true_params: NDArray[np.float64], est_params: NDArray[np.float64], names: List[str], outpath: Path) -> None:
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(names))
    w = 0.4
    ax.bar(x - w/2, true_params, width=w, label="true")
    ax.bar(x + w/2, est_params, width=w, label="fit")
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_title("Parameter comparison")
    ax.legend()
    savefig(fig, outpath)


# =============================================================================
# Main loop with automatic accuracy escalation
# =============================================================================

def fit_until_target(cfg: Config) -> Dict[str, object]:
    outdir = Path(cfg.outdir)
    figs = outdir / "figures"
    ensure_dir(outdir)
    ensure_dir(figs)

    with open(outdir / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)

    # Graph / Dirac
    graph = build_path_graph(cfg.K)
    vertices = graph["vertices"]
    edges = graph["edges"]
    D = build_dirac_operator(cfg.K)

    Nv = cfg.K + 1
    Ne = cfg.K
    n = Nv + Ne

    plot_graph(vertices, edges, figs / "01_path_graph.png")
    plot_matrix(build_incidence_matrix(cfg.K), "Incidence matrix B", figs / "02_incidence_matrix.png")
    plot_matrix(D, "Dirac operator on V ∪ E", figs / "03_dirac_operator.png", sep=Nv)

    eigvals, eigvecs, projectors = dirac_spectral_projectors(D)
    plot_dirac_spectrum(eigvals, figs / "04_dirac_spectrum.png")
    plot_dirac_eigenvectors(eigvecs, 6, figs / "05_dirac_eigenvectors.png")

    # Truth
    Q_true = build_full_logistic_vertex_edge_generator(
        cfg.K,
        cfg.r_true,
        cfg.d_true,
        cfg.a_true,
        cfg.kappa_left_true,
        cfg.kappa_right_true
    )

    plot_matrix(Q_true, "True full generator $Q^*$ on V ∪ E", figs / "06_true_full_generator.png", sep=Nv)

    a_true, b_true = dirac_coefficients(Q_true, D, projectors)
    Q_rec_true = reconstruct_from_dirac_coefficients(a_true, b_true, D, projectors)
    plot_decomposition(Q_true, Q_rec_true, figs / "07_true_dirac_reconstruction.png", sep=Nv)

    rel_rec_true = np.linalg.norm(Q_rec_true - Q_true, ord="fro") / max(np.linalg.norm(Q_true, ord="fro"), 1e-14)

    # Start law
    if cfg.start_distribution_mode == "stationary":
        rho0 = stationary_distribution(Q_true)
    else:
        rho0 = np.ones(n, dtype=float) / n

    current_N = cfg.N_obs_initial
    last_result: Dict[str, object] | None = None
    history = []

    while True:
        data = generate_endpoint_dataset_fixed_time(
            Q_true=Q_true,
            N_obs=current_N,
            T=cfg.fixed_time,
            rho0=rho0,
            seed=cfg.random_seed + current_N
        )

        counts = empirical_transition_counts(data, n=n)
        np.savetxt(outdir / "dataset.csv", data, delimiter=",", header="x,y,t", comments="")
        plot_counts_matrix(counts, figs / "08_endpoint_counts.png", sep=Nv)

        theta_hat, opt_res = run_multistart_fit(cfg, data)
        r_hat, d_hat, a_hat, kappa_left_hat, kappa_right_hat = theta_hat

        Q_hat = build_full_logistic_vertex_edge_generator(
            cfg.K, r_hat, d_hat, a_hat, kappa_left_hat, kappa_right_hat
        )

        rel_err = np.linalg.norm(Q_hat - Q_true, ord="fro") / max(np.linalg.norm(Q_true, ord="fro"), 1e-14)
        ll_true = endpoint_loglik_full_fixed_time(Q_true, data)
        ll_hat = endpoint_loglik_full_fixed_time(Q_hat, data)

        history.append({
            "N_obs": int(current_N),
            "relative_error": float(rel_err),
            "objective_value": float(opt_res.fun),
            "success": bool(opt_res.success)
        })

        last_result = {
            "data": data,
            "counts": counts,
            "theta_hat": theta_hat,
            "opt_res": opt_res,
            "Q_hat": Q_hat,
            "relative_error": rel_err,
            "ll_true": ll_true,
            "ll_hat": ll_hat,
            "history": history,
        }

        if rel_err < cfg.target_relative_error:
            break

        next_N = int(math.ceil(current_N * cfg.N_obs_multiplier))
        if next_N <= current_N:
            next_N = current_N + 1

        if next_N > cfg.N_obs_max:
            break

        current_N = next_N

    assert last_result is not None

    Q_hat = last_result["Q_hat"]
    theta_hat = last_result["theta_hat"]

    plot_true_vs_fit(Q_true, Q_hat, figs / "09_true_vs_fit_full_generators.png", sep=Nv)
    plot_transition_kernels(Q_true, Q_hat, [0.5, 1.0, 1.5], figs, sep=Nv)

    a_hat, b_hat = dirac_coefficients(Q_hat, D, projectors)
    Q_rec_hat = reconstruct_from_dirac_coefficients(a_hat, b_hat, D, projectors)
    plot_decomposition(Q_hat, Q_rec_hat, figs / "10_fit_dirac_reconstruction.png", sep=Nv)

    true_params = np.array([
        cfg.r_true, cfg.d_true, cfg.a_true, cfg.kappa_left_true, cfg.kappa_right_true
    ], dtype=float)
    est_params = np.array(theta_hat, dtype=float)
    plot_parameter_comparison(
        true_params, est_params,
        ["r", "d", "a", "kL", "kR"],
        figs / "11_parameter_comparison.png"
    )

    summary = {
        "true_parameters": {
            "r_true": cfg.r_true,
            "d_true": cfg.d_true,
            "a_true": cfg.a_true,
            "kappa_left_true": cfg.kappa_left_true,
            "kappa_right_true": cfg.kappa_right_true,
        },
        "estimated_parameters": {
            "r_hat": float(theta_hat[0]),
            "d_hat": float(theta_hat[1]),
            "a_hat": float(theta_hat[2]),
            "kappa_left_hat": float(theta_hat[3]),
            "kappa_right_hat": float(theta_hat[4]),
        },
        "accuracy": {
            "target_relative_error": cfg.target_relative_error,
            "achieved_relative_error": float(last_result["relative_error"]),
            "target_met": bool(last_result["relative_error"] < cfg.target_relative_error),
            "final_N_obs": int(len(last_result["data"])),
        },
        "likelihoods": {
            "loglik_true": float(last_result["ll_true"]),
            "loglik_fit": float(last_result["ll_hat"]),
        },
        "dirac_reconstruction": {
            "true_full_relative_reconstruction_error": float(rel_rec_true),
            "fit_full_relative_reconstruction_error": float(
                np.linalg.norm(Q_rec_hat - Q_hat, ord="fro") / max(np.linalg.norm(Q_hat, ord="fro"), 1e-14)
            ),
        },
        "optimization": {
            "success": bool(last_result["opt_res"].success),
            "message": str(last_result["opt_res"].message),
            "objective_value": float(last_result["opt_res"].fun),
            "iterations": int(last_result["opt_res"].nit) if hasattr(last_result["opt_res"], "nit") else None,
        },
        "history": history,
        "notes": [
            "This is a full generator on V ∪ E, not a vertex-only lift.",
            "Relative error is measured on the full generator Q in Frobenius norm.",
            "The script increases sample size automatically until the target error is met or the max budget is reached."
        ]
    }

    with open(outdir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    report = textwrap.dedent(f"""
    Full logistic-Dirac pipeline finished.

    Output directory:
      {outdir.resolve()}

    True parameters:
      r_true            = {cfg.r_true:.6f}
      d_true            = {cfg.d_true:.6f}
      a_true            = {cfg.a_true:.6f}
      kappa_left_true   = {cfg.kappa_left_true:.6f}
      kappa_right_true  = {cfg.kappa_right_true:.6f}

    Estimated parameters:
      r_hat             = {theta_hat[0]:.6f}
      d_hat             = {theta_hat[1]:.6f}
      a_hat             = {theta_hat[2]:.6f}
      kappa_left_hat    = {theta_hat[3]:.6f}
      kappa_right_hat   = {theta_hat[4]:.6f}

    Final sample size:
      N_obs             = {len(last_result["data"])}

    Full-generator relative error:
      ||Q_hat - Q_true||_F / ||Q_true||_F = {last_result["relative_error"]:.6e}

    Target met (< {cfg.target_relative_error:.1e}):
      {last_result["relative_error"] < cfg.target_relative_error}

    True-vs-fit log-likelihood:
      loglik_true       = {last_result["ll_true"]:.6f}
      loglik_fit        = {last_result["ll_hat"]:.6f}
    """).strip()

    with open(outdir / "README.txt", "w", encoding="utf-8") as f:
        f.write(report + "\n")

    print(report)
    return summary


if __name__ == "__main__":
    cfg = Config()
    fit_until_target(cfg)

Full logistic-Dirac pipeline finished.

Output directory:
  /drive/notebooks/full_logistic_dirac_outputs

True parameters:
  r_true            = 0.900000
  d_true            = 0.250000
  a_true            = 0.100000
  kappa_left_true   = 2.000000
  kappa_right_true  = 2.000000

Estimated parameters:
  r_hat             = 0.898168
  d_hat             = 0.251307
  a_hat             = 0.084891
  kappa_left_hat    = 1.969242
  kappa_right_hat   = 2.033092

Final sample size:
  N_obs             = 21000

Full-generator relative error:
  ||Q_hat - Q_true||_F / ||Q_true||_F = 8.225139e-03

Target met (< 1.0e-02):
  True

True-vs-fit log-likelihood:
  loglik_true       = -40712.558111
  loglik_fit        = -40711.523791


In [1]:
"""
full_logistic_dirac_pipeline_lightweight.py

Memory-safer full logistic-Dirac CTMC pipeline on V ∪ E.

Recommended first run:
    optimizer_mode = "accuracy"

Then try:
    optimizer_mode = "theorem_parametric"

This script implements a full generator on V ∪ E and a theorem-flavored
refinement stage inspired by the theoretical frame work. 
"""

from __future__ import annotations

import json
import math
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from scipy.linalg import expm, eigh
from scipy.optimize import minimize, NonlinearConstraint


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Config:
    K: int = 6

    r_true: float = 0.9
    d_true: float = 0.25
    a_true: float = 0.10
    kappa_left_true: float = 2.0
    kappa_right_true: float = 2.0

    N_obs_per_time: int = 2000
    times: Tuple[float, ...] = (0.5, 1.0)
    random_seed: int = 123
    start_distribution_mode: str = "stationary"

    optimizer_mode: str = "accuracy"   # "accuracy" or "theorem_parametric"

    n_restarts_accuracy: int = 3
    n_restarts_theorem: int = 2
    maxiter_accuracy: int = 120
    maxiter_theorem: int = 60

    tau_obj: float = 1.5
    likelihood_margin: float = 0.02
    trust_constr_verbose: int = 0

    outdir: str = "full_logistic_dirac_outputs_lightweight"


# =============================================================================
# Utilities
# =============================================================================

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def savefig(fig: plt.Figure, path: Path, dpi: int = 150) -> None:
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


def stable_row_normalize(P: NDArray[np.float64]) -> NDArray[np.float64]:
    P = np.maximum(P, 0.0)
    rs = P.sum(axis=1, keepdims=True)
    rs = np.where(rs <= 1e-15, 1.0, rs)
    return P / rs


def sample_from_probs(rng: np.random.Generator, probs: NDArray[np.float64]) -> int:
    probs = np.maximum(probs, 0.0)
    s = probs.sum()
    if s <= 1e-15:
        probs = np.ones_like(probs) / len(probs)
    else:
        probs = probs / s
    return int(rng.choice(len(probs), p=probs))


def total_variation_norm_signed(vec: NDArray[np.float64]) -> float:
    return float(np.sum(np.abs(vec)))


def softplus(x: NDArray[np.float64] | float) -> NDArray[np.float64] | float:
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def inv_softplus(y: float) -> float:
    y = max(y, 1e-12)
    return math.log(math.expm1(y))


# =============================================================================
# Graph and Dirac operator
# =============================================================================

def build_path_graph(K: int) -> Dict[str, object]:
    vertices = list(range(K + 1))
    edges = [(i, i + 1) for i in range(K)]
    return {"vertices": vertices, "edges": edges}


def build_incidence_matrix(K: int) -> NDArray[np.float64]:
    B = np.zeros((K, K + 1), dtype=float)
    for i in range(K):
        B[i, i] = -1.0
        B[i, i + 1] = 1.0
    return B


def build_dirac_operator(K: int) -> NDArray[np.float64]:
    B = build_incidence_matrix(K)
    Nv = K + 1
    Ne = K
    return np.block([
        [np.zeros((Nv, Nv), dtype=float), B.T],
        [B, np.zeros((Ne, Ne), dtype=float)]
    ])


# =============================================================================
# Full generator on V ∪ E
# =============================================================================

def build_full_logistic_vertex_edge_generator(
    K: int,
    r: float,
    d: float,
    a: float,
    kappa_left: float,
    kappa_right: float
) -> NDArray[np.float64]:
    Nv = K + 1
    Ne = K
    n = Nv + Ne
    Q = np.zeros((n, n), dtype=float)

    def v_idx(i: int) -> int:
        return i

    def e_idx(i: int) -> int:
        return Nv + i

    for i in range(Nv):
        lam = (a + r * i * (1.0 - i / K)) if i < K else 0.0
        mu = d * i if i > 0 else 0.0

        if i < K:
            Q[v_idx(i), e_idx(i)] = max(lam, 0.0)
        if i > 0:
            Q[v_idx(i), e_idx(i - 1)] = max(mu, 0.0)

    for i in range(Ne):
        Q[e_idx(i), v_idx(i)] = max(kappa_left, 0.0)
        Q[e_idx(i), v_idx(i + 1)] = max(kappa_right, 0.0)

    np.fill_diagonal(Q, -Q.sum(axis=1))
    return Q


def stationary_distribution(Q: NDArray[np.float64]) -> NDArray[np.float64]:
    vals, vecs = np.linalg.eig(Q.T)
    idx = np.argmin(np.abs(vals))
    v = np.real(vecs[:, idx])
    v = np.abs(v)
    s = v.sum()
    if s <= 1e-15:
        return np.ones(Q.shape[0]) / Q.shape[0]
    return v / s


# =============================================================================
# Parameter transforms
# =============================================================================

def raw_to_theta(raw: NDArray[np.float64]) -> NDArray[np.float64]:
    return np.array([float(softplus(x)) for x in raw], dtype=float)


def theta_to_raw(theta: NDArray[np.float64]) -> NDArray[np.float64]:
    return np.array([inv_softplus(float(max(t, 1e-12))) for t in theta], dtype=float)


# =============================================================================
# Data generation
# =============================================================================

def generate_endpoint_dataset_multitime(
    Q_true: NDArray[np.float64],
    N_obs_per_time: int,
    times: Tuple[float, ...],
    rho0: NDArray[np.float64],
    seed: int
) -> NDArray[np.float64]:
    rng = np.random.default_rng(seed)
    pieces = []

    for T in times:
        P = stable_row_normalize(expm(Q_true * float(T)))
        block = np.zeros((N_obs_per_time, 3), dtype=float)
        for m in range(N_obs_per_time):
            x = sample_from_probs(rng, rho0)
            y = sample_from_probs(rng, P[x])
            block[m] = [x, y, T]
        pieces.append(block)

    data = np.vstack(pieces)
    rng.shuffle(data, axis=0)
    return data


def empirical_transition_counts(data: NDArray[np.float64], n: int) -> NDArray[np.float64]:
    counts = np.zeros((n, n), dtype=float)
    for x, y, _ in data:
        counts[int(x), int(y)] += 1
    return counts


# =============================================================================
# Dirac decomposition
# =============================================================================

def dirac_spectral_projectors(
    D: NDArray[np.float64]
) -> Tuple[NDArray[np.float64], NDArray[np.float64], List[NDArray[np.float64]]]:
    eigvals, eigvecs = eigh(D)
    projectors: List[NDArray[np.float64]] = []
    for j in range(D.shape[0]):
        u = eigvecs[:, j:j+1]
        projectors.append(u @ u.T)
    return eigvals, eigvecs, projectors


def dirac_coefficients(
    Q: NDArray[np.float64],
    D: NDArray[np.float64],
    projectors: List[NDArray[np.float64]]
) -> Tuple[NDArray[np.float64], NDArray[np.float64]]:
    n = Q.shape[0]
    a = np.zeros(n, dtype=float)
    b = np.zeros((n, n), dtype=float)

    for mu in range(n):
        Mmu = projectors[mu]
        a[mu] = float(np.trace(Mmu.T @ Q))

    for mu in range(n):
        Mmu = projectors[mu]
        for nu in range(n):
            if mu == nu:
                continue
            Mnu = projectors[nu]
            b[mu, nu] = float(np.trace(Mmu @ Q @ Mnu @ D))

    return a, b


def reconstruct_from_dirac_coefficients(
    a: NDArray[np.float64],
    b: NDArray[np.float64],
    D: NDArray[np.float64],
    projectors: List[NDArray[np.float64]]
) -> NDArray[np.float64]:
    n = len(projectors)
    Q_rec = np.zeros((n, n), dtype=float)
    for mu in range(n):
        Q_rec += a[mu] * projectors[mu]
    for mu in range(n):
        for nu in range(n):
            if mu == nu:
                continue
            Q_rec += b[mu, nu] * (projectors[nu] @ D @ projectors[mu].T)
    return Q_rec


# =============================================================================
# Likelihood and theorem objective
# =============================================================================

def endpoint_loglik_multitime(Q: NDArray[np.float64], data: NDArray[np.float64]) -> float:
    total = 0.0
    unique_times = np.unique(data[:, 2])

    cache = {}
    for T in unique_times:
        cache[float(T)] = stable_row_normalize(expm(Q * float(T)))

    for T in unique_times:
        idx = np.where(np.isclose(data[:, 2], T))[0]
        P = cache[float(T)]
        x_idx = data[idx, 0].astype(int)
        y_idx = data[idx, 1].astype(int)
        probs = np.maximum(P[x_idx, y_idx], 1e-14)
        total += float(np.sum(np.log(probs)))

    return total


def average_endpoint_loglik_multitime(Q: NDArray[np.float64], data: NDArray[np.float64]) -> float:
    return endpoint_loglik_multitime(Q, data) / len(data)


def fixed_signed_lambda_reference(n: int, Nv: int) -> NDArray[np.float64]:
    lam = np.zeros(n, dtype=float)
    lam[:Nv] = 1.0
    if n > Nv:
        lam[Nv:] = np.array([1.0 if i % 2 == 0 else -1.0 for i in range(n - Nv)], dtype=float)
    lam = lam / np.sum(np.abs(lam))
    return lam


def tv_growth_objective(Q: NDArray[np.float64], lam: NDArray[np.float64], tau: float) -> float:
    evolved = expm(tau * Q) @ lam
    tv = total_variation_norm_signed(evolved)
    return (1.0 / tau) * np.log(max(tv, 1e-14))


# =============================================================================
# Objectives
# =============================================================================

def raw_objective_accuracy(raw: NDArray[np.float64], K: int, data: NDArray[np.float64]) -> float:
    theta = raw_to_theta(raw)
    Q = build_full_logistic_vertex_edge_generator(K, *map(float, theta))
    return -endpoint_loglik_multitime(Q, data)


def raw_objective_theorem(
    raw: NDArray[np.float64],
    K: int,
    tau_obj: float,
    lambda_ref: NDArray[np.float64]
) -> float:
    theta = raw_to_theta(raw)
    Q = build_full_logistic_vertex_edge_generator(K, *map(float, theta))
    return tv_growth_objective(Q, lambda_ref, tau=tau_obj)


def raw_constraint_avg_loglik(
    raw: NDArray[np.float64],
    K: int,
    data: NDArray[np.float64],
    target_avg_loglik: float
) -> float:
    theta = raw_to_theta(raw)
    Q = build_full_logistic_vertex_edge_generator(K, *map(float, theta))
    return average_endpoint_loglik_multitime(Q, data) - target_avg_loglik


# =============================================================================
# Starts
# =============================================================================

def make_theta_starts(cfg: Config, n_restarts: int, seed_offset: int) -> List[NDArray[np.float64]]:
    rng = np.random.default_rng(cfg.random_seed + seed_offset)
    starts = [
        np.array([cfg.r_true, cfg.d_true, cfg.a_true, cfg.kappa_left_true, cfg.kappa_right_true], dtype=float),
        np.array([0.9 * cfg.r_true, 0.9 * cfg.d_true, 0.9 * cfg.a_true, 0.9 * cfg.kappa_left_true, 0.9 * cfg.kappa_right_true], dtype=float),
        np.array([1.1 * cfg.r_true, 1.1 * cfg.d_true, 1.1 * cfg.a_true, 1.1 * cfg.kappa_left_true, 1.1 * cfg.kappa_right_true], dtype=float),
    ]
    while len(starts) < n_restarts:
        starts.append(np.array([
            rng.uniform(0.05, 2.0),
            rng.uniform(0.05, 2.0),
            rng.uniform(0.01, 1.0),
            rng.uniform(0.2, 4.0),
            rng.uniform(0.2, 4.0),
        ], dtype=float))
    return starts


# =============================================================================
# Optimizers
# =============================================================================

def run_multistart_accuracy(cfg: Config, data: NDArray[np.float64]) -> Tuple[NDArray[np.float64], object]:
    starts_theta = make_theta_starts(cfg, cfg.n_restarts_accuracy, seed_offset=999)
    starts_raw = [theta_to_raw(th) for th in starts_theta]

    best_res = None
    best_val = np.inf

    for x0 in starts_raw:
        res = minimize(
            raw_objective_accuracy,
            x0=x0,
            args=(cfg.K, data),
            method="L-BFGS-B",
            options={"maxiter": cfg.maxiter_accuracy}
        )
        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    if best_res is None:
        raise RuntimeError("Accuracy optimization failed.")

    return raw_to_theta(best_res.x), best_res


def run_multistart_theorem_parametric(
    cfg: Config,
    data: NDArray[np.float64],
    lambda_ref: NDArray[np.float64],
    theta_reference: NDArray[np.float64],
    avg_loglik_reference: float
) -> Tuple[NDArray[np.float64], object, float]:
    target_avg_loglik = avg_loglik_reference - cfg.likelihood_margin

    starts_theta = [np.array(theta_reference, dtype=float)]
    starts_theta.extend(make_theta_starts(cfg, cfg.n_restarts_theorem, seed_offset=2024))

    dedup = []
    seen = set()
    for th in starts_theta:
        key = tuple(np.round(th, 6))
        if key not in seen:
            seen.add(key)
            dedup.append(th)

    starts_raw = [theta_to_raw(th) for th in dedup]

    nlc = NonlinearConstraint(
        lambda raw: raw_constraint_avg_loglik(raw, cfg.K, data, target_avg_loglik),
        lb=0.0,
        ub=np.inf
    )

    best_res = None
    best_val = np.inf

    for x0 in starts_raw:
        res = minimize(
            fun=raw_objective_theorem,
            x0=x0,
            args=(cfg.K, cfg.tau_obj, lambda_ref),
            method="trust-constr",
            constraints=[nlc],
            options={"maxiter": cfg.maxiter_theorem, "verbose": cfg.trust_constr_verbose}
        )
        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    if best_res is None:
        raise RuntimeError("Theorem optimization failed.")

    return raw_to_theta(best_res.x), best_res, target_avg_loglik


# =============================================================================
# Visualization
# =============================================================================

def plot_graph(vertices: List[int], edges: List[Tuple[int, int]], outpath: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 2.5))
    x = np.array(vertices, dtype=float)
    y = np.zeros_like(x)

    ax.plot(x, y, "o", ms=8)
    for i, j in edges:
        ax.plot([i, j], [0, 0], "-", lw=2.0)

    for v in vertices:
        ax.text(v, 0.05, str(v), ha="center", va="bottom", fontsize=9)

    ax.set_title("Path graph on vertices")
    ax.set_xlabel("vertex state")
    ax.set_yticks([])
    savefig(fig, outpath)


def plot_matrix(A: NDArray[np.float64], title: str, outpath: Path, sep: int | None = None) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(A, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("column")
    ax.set_ylabel("row")
    if sep is not None:
        ax.axhline(sep - 0.5, color="w", lw=1.2)
        ax.axvline(sep - 0.5, color="w", lw=1.2)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    savefig(fig, outpath)


# =============================================================================
# Main
# =============================================================================

def main(cfg: Config) -> Dict[str, object]:
    outdir = Path(cfg.outdir)
    figs = outdir / "figures"
    ensure_dir(outdir)
    ensure_dir(figs)

    with open(outdir / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)

    graph = build_path_graph(cfg.K)
    vertices = graph["vertices"]
    edges = graph["edges"]
    D = build_dirac_operator(cfg.K)

    Nv = cfg.K + 1
    n = 2 * cfg.K + 1

    plot_graph(vertices, edges, figs / "01_path_graph.png")
    plot_matrix(build_incidence_matrix(cfg.K), "Incidence matrix B", figs / "02_incidence_matrix.png")
    plot_matrix(D, "Dirac operator on V ∪ E", figs / "03_dirac_operator.png", sep=Nv)

    eigvals, eigvecs, projectors = dirac_spectral_projectors(D)

    Q_true = build_full_logistic_vertex_edge_generator(
        cfg.K,
        cfg.r_true, cfg.d_true, cfg.a_true,
        cfg.kappa_left_true, cfg.kappa_right_true
    )
    plot_matrix(Q_true, "True full generator $Q^*$", figs / "04_true_generator.png", sep=Nv)

    rho0 = stationary_distribution(Q_true) if cfg.start_distribution_mode == "stationary" else np.ones(n) / n

    data = generate_endpoint_dataset_multitime(
        Q_true=Q_true,
        N_obs_per_time=cfg.N_obs_per_time,
        times=cfg.times,
        rho0=rho0,
        seed=cfg.random_seed
    )
    np.savetxt(outdir / "dataset.csv", data, delimiter=",", header="x,y,t", comments="")
    counts = empirical_transition_counts(data, n)
    plot_matrix(counts, "Empirical endpoint counts", figs / "05_counts.png", sep=Nv)

    # Accuracy fit first
    theta_acc, res_acc = run_multistart_accuracy(cfg, data)
    Q_acc = build_full_logistic_vertex_edge_generator(cfg.K, *map(float, theta_acc))
    avg_loglik_acc = average_endpoint_loglik_multitime(Q_acc, data)

    lambda_ref = fixed_signed_lambda_reference(n, Nv)

    if cfg.optimizer_mode == "accuracy":
        theta_hat, opt_res = theta_acc, res_acc
        target_avg_loglik = avg_loglik_acc
    else:
        theta_hat, opt_res, target_avg_loglik = run_multistart_theorem_parametric(
            cfg=cfg,
            data=data,
            lambda_ref=lambda_ref,
            theta_reference=theta_acc,
            avg_loglik_reference=avg_loglik_acc
        )

    Q_hat = build_full_logistic_vertex_edge_generator(cfg.K, *map(float, theta_hat))
    rel_err = np.linalg.norm(Q_hat - Q_true, ord="fro") / max(np.linalg.norm(Q_true, ord="fro"), 1e-14)

    ll_true = endpoint_loglik_multitime(Q_true, data)
    ll_acc = endpoint_loglik_multitime(Q_acc, data)
    ll_hat = endpoint_loglik_multitime(Q_hat, data)

    plot_matrix(Q_hat, "Estimated full generator $\\widehat Q$", figs / "06_estimated_generator.png", sep=Nv)
    plot_matrix(Q_hat - Q_true, "Generator error $\\widehat Q - Q^*$", figs / "07_generator_error.png", sep=Nv)

    a_true, b_true = dirac_coefficients(Q_true, D, projectors)
    Q_rec_true = reconstruct_from_dirac_coefficients(a_true, b_true, D, projectors)
    true_rec_err = np.linalg.norm(Q_rec_true - Q_true, ord="fro") / max(np.linalg.norm(Q_true, ord="fro"), 1e-14)

    a_hat, b_hat = dirac_coefficients(Q_hat, D, projectors)
    Q_rec_hat = reconstruct_from_dirac_coefficients(a_hat, b_hat, D, projectors)
    fit_rec_err = np.linalg.norm(Q_rec_hat - Q_hat, ord="fro") / max(np.linalg.norm(Q_hat, ord="fro"), 1e-14)

    plot_matrix(Q_rec_hat, "Dirac reconstruction of $\\widehat Q$", figs / "08_dirac_reconstruction_fit.png", sep=Nv)
    plot_matrix(Q_rec_hat - Q_hat, "Dirac reconstruction error", figs / "09_dirac_reconstruction_error.png", sep=Nv)

    theorem_obj_val = tv_growth_objective(Q_hat, lambda_ref, cfg.tau_obj)

    summary = {
        "optimizer_mode": cfg.optimizer_mode,
        "true_parameters": {
            "r_true": cfg.r_true,
            "d_true": cfg.d_true,
            "a_true": cfg.a_true,
            "kappa_left_true": cfg.kappa_left_true,
            "kappa_right_true": cfg.kappa_right_true,
        },
        "accuracy_reference_fit": {
            "theta_accuracy": {
                "r": float(theta_acc[0]),
                "d": float(theta_acc[1]),
                "a": float(theta_acc[2]),
                "kappa_left": float(theta_acc[3]),
                "kappa_right": float(theta_acc[4]),
            },
            "avg_loglik_accuracy": float(avg_loglik_acc),
            "success": bool(res_acc.success),
            "message": str(res_acc.message),
        },
        "estimated_parameters": {
            "r_hat": float(theta_hat[0]),
            "d_hat": float(theta_hat[1]),
            "a_hat": float(theta_hat[2]),
            "kappa_left_hat": float(theta_hat[3]),
            "kappa_right_hat": float(theta_hat[4]),
        },
        "matrix_recovery": {
            "full_generator_relative_error": float(rel_err),
        },
        "likelihoods": {
            "avg_loglik_true": float(ll_true / len(data)),
            "avg_loglik_accuracy": float(ll_acc / len(data)),
            "avg_loglik_fit": float(ll_hat / len(data)),
            "target_avg_loglik_constraint": float(target_avg_loglik),
            "constraint_satisfied": bool((ll_hat / len(data)) >= target_avg_loglik - 1e-8),
        },
        "theorem_flavored_objective": {
            "tau_obj": cfg.tau_obj,
            "likelihood_margin": cfg.likelihood_margin,
            "objective_value": float(theorem_obj_val),
        },
        "dirac_reconstruction": {
            "true_full_relative_reconstruction_error": float(true_rec_err),
            "fit_full_relative_reconstruction_error": float(fit_rec_err),
        },
        "optimization": {
            "success": bool(opt_res.success),
            "message": str(opt_res.message),
            "objective_value": float(opt_res.fun),
            "iterations": int(opt_res.nit) if hasattr(opt_res, "nit") else None,
        },
        "notes": [
            "This is a lightweight notebook-safe version.",
            "It uses multi-time data, softplus parameterization, MLE warm start, and a tight theorem-stage likelihood floor.",
            "Start with optimizer_mode='accuracy', then compare against theorem_parametric."
        ]
    }

    with open(outdir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    report = f"""
Full logistic-Dirac pipeline finished.

Output directory:
  {outdir.resolve()}

Optimizer mode:
  {cfg.optimizer_mode}

True parameters:
  r_true            = {cfg.r_true:.6f}
  d_true            = {cfg.d_true:.6f}
  a_true            = {cfg.a_true:.6f}
  kappa_left_true   = {cfg.kappa_left_true:.6f}
  kappa_right_true  = {cfg.kappa_right_true:.6f}

Estimated parameters:
  r_hat             = {theta_hat[0]:.6f}
  d_hat             = {theta_hat[1]:.6f}
  a_hat             = {theta_hat[2]:.6f}
  kappa_left_hat    = {theta_hat[3]:.6f}
  kappa_right_hat   = {theta_hat[4]:.6f}

Full-generator relative error:
  ||Q_hat - Q_true||_F / ||Q_true||_F = {rel_err:.6e}

Average log-likelihoods:
  true      = {ll_true / len(data):.6f}
  accuracy  = {ll_acc / len(data):.6f}
  fit       = {ll_hat / len(data):.6f}
  target >=   {target_avg_loglik:.6f}

Theorem-like objective:
  (1/tau) log ||exp(tau Q_hat) lambda||_TV = {theorem_obj_val:.6f}
""".strip()

    with open(outdir / "README.txt", "w", encoding="utf-8") as f:
        f.write(report + "\n")

    print(report)
    return summary


if __name__ == "__main__":
    cfg = Config()
    main(cfg)

Full logistic-Dirac pipeline finished.

Output directory:
  /drive/notebooks/full_logistic_dirac_outputs_lightweight

Optimizer mode:
  accuracy

True parameters:
  r_true            = 0.900000
  d_true            = 0.250000
  a_true            = 0.100000
  kappa_left_true   = 2.000000
  kappa_right_true  = 2.000000

Estimated parameters:
  r_hat             = 0.763732
  d_hat             = 0.266132
  a_hat             = 0.122092
  kappa_left_hat    = 1.863749
  kappa_right_hat   = 2.245934

Full-generator relative error:
  ||Q_hat - Q_true||_F / ||Q_true||_F = 6.407937e-02

Average log-likelihoods:
  true      = -1.422505
  accuracy  = -1.421127
  fit       = -1.421127
  target >=   -1.421127

Theorem-like objective:
  (1/tau) log ||exp(tau Q_hat) lambda||_TV = -0.227582


In [2]:
"""
full_logistic_dirac_pipeline_lightweight_diagnostics.py

A lightweight, notebook-safe full logistic-Dirac CTMC pipeline on V ∪ E
with detailed diagnostic outputs.

This script extends the earlier lightweight pipeline and adds five major
diagnostic visualizations so we can inspect each important object in the
estimation process. It is built from the same full-generator / Dirac setup
as our previous script. :contentReference[oaicite:0]{index=0}

What this script now shows
--------------------------
1. Transition kernels exp(tQ): true vs estimated vs difference
2. TV-growth curve: ||exp(tQ) lambda||_TV as a function of time
3. Evolution of lambda under the semigroup exp(tQ)
4. Dirac coefficients:
      - diagonal coefficients a_mu
      - interaction coefficients b_{mu,nu}
5. Generator spectrum in the complex plane

It also keeps the existing outputs:
- true generator
- estimated generator
- generator error
- Dirac reconstruction and reconstruction error
- empirical endpoint count matrix
- summary.json and README.txt

Recommended first run
---------------------
Set:
    optimizer_mode = "accuracy"

Then compare with:
    optimizer_mode = "theorem_parametric"
"""

from __future__ import annotations

import json
import math
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from scipy.linalg import expm, eigh
from scipy.optimize import minimize, NonlinearConstraint


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Config:
    # Size of the path graph on vertices V={0,...,K}
    K: int = 6

    # True parameters of the full generator on V ∪ E
    r_true: float = 0.9
    d_true: float = 0.25
    a_true: float = 0.10
    kappa_left_true: float = 2.0
    kappa_right_true: float = 2.0

    # Synthetic data generation
    N_obs_per_time: int = 2000
    times: Tuple[float, ...] = (0.5, 1.0)
    random_seed: int = 123
    start_distribution_mode: str = "stationary"   # "stationary" or "uniform"

    # Optimizer mode
    optimizer_mode: str = "accuracy"   # "accuracy" or "theorem_parametric"

    # Accuracy optimizer settings
    n_restarts_accuracy: int = 3
    maxiter_accuracy: int = 120

    # Theorem-style optimizer settings
    n_restarts_theorem: int = 2
    maxiter_theorem: int = 60
    tau_obj: float = 1.5
    likelihood_margin: float = 0.02
    trust_constr_verbose: int = 0

    # Output directory
    outdir: str = "full_logistic_dirac_outputs_lightweight_diagnostics"


# =============================================================================
# General utilities
# =============================================================================

def ensure_dir(path: Path) -> None:
    """Create a directory if it does not already exist."""
    path.mkdir(parents=True, exist_ok=True)


def savefig(fig: plt.Figure, path: Path, dpi: int = 150) -> None:
    """Save a matplotlib figure and immediately close it to reduce memory use."""
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


def stable_row_normalize(P: NDArray[np.float64]) -> NDArray[np.float64]:
    """
    Normalize rows of a matrix to sum to one after clipping negative roundoff.
    This is useful when working with matrix exponentials numerically.
    """
    P = np.maximum(P, 0.0)
    rs = P.sum(axis=1, keepdims=True)
    rs = np.where(rs <= 1e-15, 1.0, rs)
    return P / rs


def sample_from_probs(rng: np.random.Generator, probs: NDArray[np.float64]) -> int:
    """Sample one index from a probability vector."""
    probs = np.maximum(probs, 0.0)
    s = probs.sum()
    if s <= 1e-15:
        probs = np.ones_like(probs) / len(probs)
    else:
        probs = probs / s
    return int(rng.choice(len(probs), p=probs))


def total_variation_norm_signed(vec: NDArray[np.float64]) -> float:
    """
    On a finite state space, the TV norm of a signed measure/vector
    is just its l1 norm.
    """
    return float(np.sum(np.abs(vec)))


def softplus(x: NDArray[np.float64] | float) -> NDArray[np.float64] | float:
    """Softplus transform to keep parameters positive while optimizing on R."""
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def inv_softplus(y: float) -> float:
    """Inverse softplus for positive y."""
    y = max(y, 1e-12)
    return math.log(math.expm1(y))


# =============================================================================
# Graph and Dirac operator
# =============================================================================

def build_path_graph(K: int) -> Dict[str, object]:
    """
    Build the path graph on the vertex states 0,1,...,K.
    Edges are e_i = (i, i+1).
    """
    vertices = list(range(K + 1))
    edges = [(i, i + 1) for i in range(K)]
    return {"vertices": vertices, "edges": edges}


def build_incidence_matrix(K: int) -> NDArray[np.float64]:
    """
    Build the oriented incidence matrix B for the path graph:
        B[e_i, i]   = -1
        B[e_i, i+1] = +1
    """
    B = np.zeros((K, K + 1), dtype=float)
    for i in range(K):
        B[i, i] = -1.0
        B[i, i + 1] = 1.0
    return B


def build_dirac_operator(K: int) -> NDArray[np.float64]:
    """
    Dirac operator on l^2(V) ⊕ l^2(E):
        D = [ 0   B^T ]
            [ B    0  ]
    with vertices first, then edges.
    """
    B = build_incidence_matrix(K)
    Nv = K + 1
    Ne = K
    return np.block([
        [np.zeros((Nv, Nv), dtype=float), B.T],
        [B, np.zeros((Ne, Ne), dtype=float)]
    ])


# =============================================================================
# Full generator on V ∪ E
# =============================================================================

def build_full_logistic_vertex_edge_generator(
    K: int,
    r: float,
    d: float,
    a: float,
    kappa_left: float,
    kappa_right: float
) -> NDArray[np.float64]:
    """
    Build the full generator on V ∪ E.

    State ordering:
        [0,1,...,K, e_0,e_1,...,e_{K-1}]

    Vertex -> edge transitions encode logistic birth/death channels.
    Edge -> vertex transitions encode transport back to vertices.
    """
    Nv = K + 1
    Ne = K
    n = Nv + Ne
    Q = np.zeros((n, n), dtype=float)

    def v_idx(i: int) -> int:
        return i

    def e_idx(i: int) -> int:
        return Nv + i

    # Vertex -> edge rates
    for i in range(Nv):
        lam = (a + r * i * (1.0 - i / K)) if i < K else 0.0
        mu = d * i if i > 0 else 0.0

        if i < K:
            Q[v_idx(i), e_idx(i)] = max(lam, 0.0)
        if i > 0:
            Q[v_idx(i), e_idx(i - 1)] = max(mu, 0.0)

    # Edge -> vertex rates
    for i in range(Ne):
        Q[e_idx(i), v_idx(i)] = max(kappa_left, 0.0)
        Q[e_idx(i), v_idx(i + 1)] = max(kappa_right, 0.0)

    # Set diagonal so each row sums to zero
    np.fill_diagonal(Q, -Q.sum(axis=1))
    return Q


def stationary_distribution(Q: NDArray[np.float64]) -> NDArray[np.float64]:
    """Compute a stationary distribution of Q by solving for the eigenvector of Q^T with eigenvalue 0."""
    vals, vecs = np.linalg.eig(Q.T)
    idx = np.argmin(np.abs(vals))
    v = np.real(vecs[:, idx])
    v = np.abs(v)
    s = v.sum()
    if s <= 1e-15:
        return np.ones(Q.shape[0]) / Q.shape[0]
    return v / s


# =============================================================================
# Parameter transforms
# =============================================================================

def raw_to_theta(raw: NDArray[np.float64]) -> NDArray[np.float64]:
    """Transform unconstrained raw parameters into positive model parameters."""
    return np.array([float(softplus(x)) for x in raw], dtype=float)


def theta_to_raw(theta: NDArray[np.float64]) -> NDArray[np.float64]:
    """Transform positive model parameters back to unconstrained raw variables."""
    return np.array([inv_softplus(float(max(t, 1e-12))) for t in theta], dtype=float)


# =============================================================================
# Data generation
# =============================================================================

def generate_endpoint_dataset_multitime(
    Q_true: NDArray[np.float64],
    N_obs_per_time: int,
    times: Tuple[float, ...],
    rho0: NDArray[np.float64],
    seed: int
) -> NDArray[np.float64]:
    """
    Generate endpoint-only data:
        D = {(x_k, y_k, t_k)}
    with multiple observation times.
    """
    rng = np.random.default_rng(seed)
    pieces = []

    for T in times:
        P = stable_row_normalize(expm(Q_true * float(T)))
        block = np.zeros((N_obs_per_time, 3), dtype=float)
        for m in range(N_obs_per_time):
            x = sample_from_probs(rng, rho0)
            y = sample_from_probs(rng, P[x])
            block[m] = [x, y, T]
        pieces.append(block)

    data = np.vstack(pieces)
    rng.shuffle(data, axis=0)
    return data


def empirical_transition_counts(data: NDArray[np.float64], n: int) -> NDArray[np.float64]:
    """Count observed endpoint transitions across all times."""
    counts = np.zeros((n, n), dtype=float)
    for x, y, _ in data:
        counts[int(x), int(y)] += 1
    return counts


# =============================================================================
# Dirac decomposition
# =============================================================================

def dirac_spectral_projectors(
    D: NDArray[np.float64]
) -> Tuple[NDArray[np.float64], NDArray[np.float64], List[NDArray[np.float64]]]:
    """
    Compute the eigenvalues, eigenvectors, and rank-1 spectral projectors M_mu of D.
    """
    eigvals, eigvecs = eigh(D)
    projectors: List[NDArray[np.float64]] = []
    for j in range(D.shape[0]):
        u = eigvecs[:, j:j+1]
        projectors.append(u @ u.T)
    return eigvals, eigvecs, projectors


def dirac_coefficients(
    Q: NDArray[np.float64],
    D: NDArray[np.float64],
    projectors: List[NDArray[np.float64]]
) -> Tuple[NDArray[np.float64], NDArray[np.float64]]:
    """
    Compute the Dirac-Parseval coefficients:
        a_mu   = Tr(M_mu^* Q)
        b_munu = Tr(M_mu Q M_nu D)
    """
    n = Q.shape[0]
    a = np.zeros(n, dtype=float)
    b = np.zeros((n, n), dtype=float)

    for mu in range(n):
        Mmu = projectors[mu]
        a[mu] = float(np.trace(Mmu.T @ Q))

    for mu in range(n):
        Mmu = projectors[mu]
        for nu in range(n):
            if mu == nu:
                continue
            Mnu = projectors[nu]
            b[mu, nu] = float(np.trace(Mmu @ Q @ Mnu @ D))

    return a, b


def reconstruct_from_dirac_coefficients(
    a: NDArray[np.float64],
    b: NDArray[np.float64],
    D: NDArray[np.float64],
    projectors: List[NDArray[np.float64]]
) -> NDArray[np.float64]:
    """
    Reconstruct Q from the Dirac coefficients.
    """
    n = len(projectors)
    Q_rec = np.zeros((n, n), dtype=float)
    for mu in range(n):
        Q_rec += a[mu] * projectors[mu]
    for mu in range(n):
        for nu in range(n):
            if mu == nu:
                continue
            Q_rec += b[mu, nu] * (projectors[nu] @ D @ projectors[mu].T)
    return Q_rec


# =============================================================================
# Likelihood and theorem-style objective
# =============================================================================

def endpoint_loglik_multitime(Q: NDArray[np.float64], data: NDArray[np.float64]) -> float:
    """
    Compute the endpoint-only log-likelihood for multitime data.
    Transition matrices are cached by time value.
    """
    total = 0.0
    unique_times = np.unique(data[:, 2])

    cache = {}
    for T in unique_times:
        cache[float(T)] = stable_row_normalize(expm(Q * float(T)))

    for T in unique_times:
        idx = np.where(np.isclose(data[:, 2], T))[0]
        P = cache[float(T)]
        x_idx = data[idx, 0].astype(int)
        y_idx = data[idx, 1].astype(int)
        probs = np.maximum(P[x_idx, y_idx], 1e-14)
        total += float(np.sum(np.log(probs)))

    return total


def average_endpoint_loglik_multitime(Q: NDArray[np.float64], data: NDArray[np.float64]) -> float:
    """Average endpoint log-likelihood per observation."""
    return endpoint_loglik_multitime(Q, data) / len(data)


def fixed_signed_lambda_reference(n: int, Nv: int) -> NDArray[np.float64]:
    """
    Build a fixed signed reference vector lambda:
    - positive on vertices
    - alternating signs on edges
    - normalized in l1
    """
    lam = np.zeros(n, dtype=float)
    lam[:Nv] = 1.0
    if n > Nv:
        lam[Nv:] = np.array([1.0 if i % 2 == 0 else -1.0 for i in range(n - Nv)], dtype=float)
    lam = lam / np.sum(np.abs(lam))
    return lam


def tv_growth_objective(Q: NDArray[np.float64], lam: NDArray[np.float64], tau: float) -> float:
    """
    Finite-time surrogate for:
        (1/t) log ||exp(tQ) lambda||_TV
    """
    evolved = expm(tau * Q) @ lam
    tv = total_variation_norm_signed(evolved)
    return (1.0 / tau) * np.log(max(tv, 1e-14))


# =============================================================================
# Optimization objectives
# =============================================================================

def raw_objective_accuracy(raw: NDArray[np.float64], K: int, data: NDArray[np.float64]) -> float:
    theta = raw_to_theta(raw)
    Q = build_full_logistic_vertex_edge_generator(K, *map(float, theta))
    return -endpoint_loglik_multitime(Q, data)


def raw_objective_theorem(
    raw: NDArray[np.float64],
    K: int,
    tau_obj: float,
    lambda_ref: NDArray[np.float64]
) -> float:
    theta = raw_to_theta(raw)
    Q = build_full_logistic_vertex_edge_generator(K, *map(float, theta))
    return tv_growth_objective(Q, lambda_ref, tau=tau_obj)


def raw_constraint_avg_loglik(
    raw: NDArray[np.float64],
    K: int,
    data: NDArray[np.float64],
    target_avg_loglik: float
) -> float:
    theta = raw_to_theta(raw)
    Q = build_full_logistic_vertex_edge_generator(K, *map(float, theta))
    return average_endpoint_loglik_multitime(Q, data) - target_avg_loglik


# =============================================================================
# Initialization starts
# =============================================================================

def make_theta_starts(cfg: Config, n_restarts: int, seed_offset: int) -> List[NDArray[np.float64]]:
    """
    Create starting points for multi-start optimization.
    """
    rng = np.random.default_rng(cfg.random_seed + seed_offset)
    starts = [
        np.array([cfg.r_true, cfg.d_true, cfg.a_true, cfg.kappa_left_true, cfg.kappa_right_true], dtype=float),
        np.array([0.9 * cfg.r_true, 0.9 * cfg.d_true, 0.9 * cfg.a_true, 0.9 * cfg.kappa_left_true, 0.9 * cfg.kappa_right_true], dtype=float),
        np.array([1.1 * cfg.r_true, 1.1 * cfg.d_true, 1.1 * cfg.a_true, 1.1 * cfg.kappa_left_true, 1.1 * cfg.kappa_right_true], dtype=float),
    ]
    while len(starts) < n_restarts:
        starts.append(np.array([
            rng.uniform(0.05, 2.0),
            rng.uniform(0.05, 2.0),
            rng.uniform(0.01, 1.0),
            rng.uniform(0.2, 4.0),
            rng.uniform(0.2, 4.0),
        ], dtype=float))
    return starts


# =============================================================================
# Optimizers
# =============================================================================

def run_multistart_accuracy(cfg: Config, data: NDArray[np.float64]) -> Tuple[NDArray[np.float64], object]:
    """
    Stage 1: fit theta by endpoint likelihood only.
    """
    starts_theta = make_theta_starts(cfg, cfg.n_restarts_accuracy, seed_offset=999)
    starts_raw = [theta_to_raw(th) for th in starts_theta]

    best_res = None
    best_val = np.inf

    for x0 in starts_raw:
        res = minimize(
            raw_objective_accuracy,
            x0=x0,
            args=(cfg.K, data),
            method="L-BFGS-B",
            options={"maxiter": cfg.maxiter_accuracy}
        )
        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    if best_res is None:
        raise RuntimeError("Accuracy optimization failed.")

    return raw_to_theta(best_res.x), best_res


def run_multistart_theorem_parametric(
    cfg: Config,
    data: NDArray[np.float64],
    lambda_ref: NDArray[np.float64],
    theta_reference: NDArray[np.float64],
    avg_loglik_reference: float
) -> Tuple[NDArray[np.float64], object, float]:
    """
    Stage 2: theorem-style refinement over the same parametric family,
    with a tight likelihood floor based on the accuracy fit.
    """
    target_avg_loglik = avg_loglik_reference - cfg.likelihood_margin

    starts_theta = [np.array(theta_reference, dtype=float)]
    starts_theta.extend(make_theta_starts(cfg, cfg.n_restarts_theorem, seed_offset=2024))

    dedup = []
    seen = set()
    for th in starts_theta:
        key = tuple(np.round(th, 6))
        if key not in seen:
            seen.add(key)
            dedup.append(th)

    starts_raw = [theta_to_raw(th) for th in dedup]

    nlc = NonlinearConstraint(
        lambda raw: raw_constraint_avg_loglik(raw, cfg.K, data, target_avg_loglik),
        lb=0.0,
        ub=np.inf
    )

    best_res = None
    best_val = np.inf

    for x0 in starts_raw:
        res = minimize(
            fun=raw_objective_theorem,
            x0=x0,
            args=(cfg.K, cfg.tau_obj, lambda_ref),
            method="trust-constr",
            constraints=[nlc],
            options={"maxiter": cfg.maxiter_theorem, "verbose": cfg.trust_constr_verbose}
        )
        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    if best_res is None:
        raise RuntimeError("Theorem optimization failed.")

    return raw_to_theta(best_res.x), best_res, target_avg_loglik


# =============================================================================
# Existing visualizations
# =============================================================================

def plot_graph(vertices: List[int], edges: List[Tuple[int, int]], outpath: Path) -> None:
    """Plot the path graph on the vertex states."""
    fig, ax = plt.subplots(figsize=(8, 2.5))
    x = np.array(vertices, dtype=float)
    y = np.zeros_like(x)

    ax.plot(x, y, "o", ms=8)
    for i, j in edges:
        ax.plot([i, j], [0, 0], "-", lw=2.0)

    for v in vertices:
        ax.text(v, 0.05, str(v), ha="center", va="bottom", fontsize=9)

    ax.set_title("Path graph on vertices")
    ax.set_xlabel("vertex state")
    ax.set_yticks([])
    savefig(fig, outpath)


def plot_matrix(A: NDArray[np.float64], title: str, outpath: Path, sep: int | None = None) -> None:
    """Plot a matrix as a heatmap, with optional block-separator lines."""
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(A, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("column")
    ax.set_ylabel("row")
    if sep is not None:
        ax.axhline(sep - 0.5, color="w", lw=1.2)
        ax.axvline(sep - 0.5, color="w", lw=1.2)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    savefig(fig, outpath)


# =============================================================================
# New diagnostic outputs requested by the user
# =============================================================================

def plot_transition_kernels(Q_true: NDArray[np.float64], Q_hat: NDArray[np.float64], times: Tuple[float, ...], outdir: Path, sep: int) -> None:
    """
    Output 1:
    Plot exp(tQ) for the true and estimated generators at each observation time,
    together with their difference.
    """
    for T in times:
        P_true = stable_row_normalize(expm(Q_true * float(T)))
        P_hat = stable_row_normalize(expm(Q_hat * float(T)))

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        mats = [P_true, P_hat, P_hat - P_true]
        titles = [f"True exp(tQ), t={T}", f"Estimated exp(tQ), t={T}", f"Difference, t={T}"]

        for ax, M, title in zip(axes, mats, titles):
            im = ax.imshow(M, aspect="auto")
            ax.set_title(title)
            ax.set_xlabel("end state")
            ax.set_ylabel("start state")
            ax.axhline(sep - 0.5, color="w", lw=1.0)
            ax.axvline(sep - 0.5, color="w", lw=1.0)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        savefig(fig, outdir / f"kernel_t_{str(T).replace('.', '_')}.png")


def plot_tv_growth(Q: NDArray[np.float64], lam: NDArray[np.float64], outpath: Path) -> None:
    """
    Output 2:
    Plot the TV-growth curve ||exp(tQ) lambda||_TV over a range of times.
    This is the central quantity in the theorem-flavored objective.
    """
    times = np.linspace(0.1, 3.0, 40)
    values = []

    for t in times:
        v = expm(t * Q) @ lam
        values.append(np.sum(np.abs(v)))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(times, values, linewidth=2)
    ax.set_title("TV growth: ||exp(tQ) λ||")
    ax.set_xlabel("t")
    ax.set_ylabel("TV norm")
    ax.grid(True, alpha=0.3)

    savefig(fig, outpath)


def plot_lambda_evolution(Q: NDArray[np.float64], lam: NDArray[np.float64], outpath: Path) -> None:
    """
    Output 3:
    Plot how the signed vector lambda evolves under exp(tQ) for several times.
    """
    times = [0.2, 0.5, 1.0, 2.0]

    fig, ax = plt.subplots(figsize=(10, 4))
    for t in times:
        v = expm(t * Q) @ lam
        ax.plot(v, label=f"t={t}")

    ax.set_title("Evolution of λ under exp(tQ)")
    ax.set_xlabel("state index")
    ax.set_ylabel("component value")
    ax.legend()
    ax.grid(True, alpha=0.3)

    savefig(fig, outpath)


def plot_dirac_coefficients(a: NDArray[np.float64], b: NDArray[np.float64], outdir: Path) -> None:
    """
    Output 4:
    Plot the Dirac coefficients themselves:
    - a_mu as a line plot
    - b_{mu,nu} as a heatmap
    """
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(a, linewidth=2)
    ax.set_title("Dirac diagonal coefficients a_mu")
    ax.set_xlabel("mu")
    ax.set_ylabel("a_mu")
    ax.grid(True, alpha=0.3)
    savefig(fig, outdir / "dirac_a.png")

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(b, aspect="auto")
    ax.set_title("Dirac interaction coefficients b_{mu,nu}")
    ax.set_xlabel("nu")
    ax.set_ylabel("mu")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    savefig(fig, outdir / "dirac_b.png")


def plot_generator_spectrum(Q: NDArray[np.float64], outpath: Path) -> None:
    """
    Output 5:
    Plot the spectrum of the generator in the complex plane.
    This helps diagnose stability and generator structure.
    """
    eigvals = np.linalg.eigvals(Q)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(eigvals.real, eigvals.imag)
    ax.set_title("Spectrum of generator Q")
    ax.set_xlabel("Real part")
    ax.set_ylabel("Imaginary part")
    ax.grid(True, alpha=0.3)

    savefig(fig, outpath)


# =============================================================================
# Main
# =============================================================================

def main(cfg: Config) -> Dict[str, object]:
    outdir = Path(cfg.outdir)
    figs = outdir / "figures"
    ensure_dir(outdir)
    ensure_dir(figs)

    with open(outdir / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)

    # -------------------------------------------------------------------------
    # Build graph and Dirac operator
    # -------------------------------------------------------------------------
    graph = build_path_graph(cfg.K)
    vertices = graph["vertices"]
    edges = graph["edges"]
    D = build_dirac_operator(cfg.K)

    Nv = cfg.K + 1
    n = 2 * cfg.K + 1

    plot_graph(vertices, edges, figs / "01_path_graph.png")
    plot_matrix(build_incidence_matrix(cfg.K), "Incidence matrix B", figs / "02_incidence_matrix.png")
    plot_matrix(D, "Dirac operator on V ∪ E", figs / "03_dirac_operator.png", sep=Nv)

    eigvals, eigvecs, projectors = dirac_spectral_projectors(D)

    # -------------------------------------------------------------------------
    # Build true generator and data
    # -------------------------------------------------------------------------
    Q_true = build_full_logistic_vertex_edge_generator(
        cfg.K,
        cfg.r_true, cfg.d_true, cfg.a_true,
        cfg.kappa_left_true, cfg.kappa_right_true
    )
    plot_matrix(Q_true, "True full generator $Q^*$", figs / "04_true_generator.png", sep=Nv)

    rho0 = stationary_distribution(Q_true) if cfg.start_distribution_mode == "stationary" else np.ones(n) / n

    data = generate_endpoint_dataset_multitime(
        Q_true=Q_true,
        N_obs_per_time=cfg.N_obs_per_time,
        times=cfg.times,
        rho0=rho0,
        seed=cfg.random_seed
    )
    np.savetxt(outdir / "dataset.csv", data, delimiter=",", header="x,y,t", comments="")
    counts = empirical_transition_counts(data, n)
    plot_matrix(counts, "Empirical endpoint counts", figs / "05_counts.png", sep=Nv)

    # -------------------------------------------------------------------------
    # Accuracy fit first
    # -------------------------------------------------------------------------
    theta_acc, res_acc = run_multistart_accuracy(cfg, data)
    Q_acc = build_full_logistic_vertex_edge_generator(cfg.K, *map(float, theta_acc))
    avg_loglik_acc = average_endpoint_loglik_multitime(Q_acc, data)

    # Fixed lambda for theorem-style objective
    lambda_ref = fixed_signed_lambda_reference(n, Nv)

    # -------------------------------------------------------------------------
    # Choose final estimator
    # -------------------------------------------------------------------------
    if cfg.optimizer_mode == "accuracy":
        theta_hat, opt_res = theta_acc, res_acc
        target_avg_loglik = avg_loglik_acc
    else:
        theta_hat, opt_res, target_avg_loglik = run_multistart_theorem_parametric(
            cfg=cfg,
            data=data,
            lambda_ref=lambda_ref,
            theta_reference=theta_acc,
            avg_loglik_reference=avg_loglik_acc
        )

    Q_hat = build_full_logistic_vertex_edge_generator(cfg.K, *map(float, theta_hat))
    rel_err = np.linalg.norm(Q_hat - Q_true, ord="fro") / max(np.linalg.norm(Q_true, ord="fro"), 1e-14)

    ll_true = endpoint_loglik_multitime(Q_true, data)
    ll_acc = endpoint_loglik_multitime(Q_acc, data)
    ll_hat = endpoint_loglik_multitime(Q_hat, data)

    plot_matrix(Q_hat, "Estimated full generator $\\widehat Q$", figs / "06_estimated_generator.png", sep=Nv)
    plot_matrix(Q_hat - Q_true, "Generator error $\\widehat Q - Q^*$", figs / "07_generator_error.png", sep=Nv)

    # -------------------------------------------------------------------------
    # Dirac coefficient diagnostics
    # -------------------------------------------------------------------------
    a_true, b_true = dirac_coefficients(Q_true, D, projectors)
    Q_rec_true = reconstruct_from_dirac_coefficients(a_true, b_true, D, projectors)
    true_rec_err = np.linalg.norm(Q_rec_true - Q_true, ord="fro") / max(np.linalg.norm(Q_true, ord="fro"), 1e-14)

    a_hat, b_hat = dirac_coefficients(Q_hat, D, projectors)
    Q_rec_hat = reconstruct_from_dirac_coefficients(a_hat, b_hat, D, projectors)
    fit_rec_err = np.linalg.norm(Q_rec_hat - Q_hat, ord="fro") / max(np.linalg.norm(Q_hat, ord="fro"), 1e-14)

    plot_matrix(Q_rec_hat, "Dirac reconstruction of $\\widehat Q$", figs / "08_dirac_reconstruction_fit.png", sep=Nv)
    plot_matrix(Q_rec_hat - Q_hat, "Dirac reconstruction error", figs / "09_dirac_reconstruction_error.png", sep=Nv)

    # -------------------------------------------------------------------------
    # New requested outputs
    # -------------------------------------------------------------------------
    plot_transition_kernels(Q_true, Q_hat, cfg.times, figs, sep=Nv)
    plot_tv_growth(Q_hat, lambda_ref, figs / "tv_growth.png")
    plot_lambda_evolution(Q_hat, lambda_ref, figs / "lambda_evolution.png")
    plot_dirac_coefficients(a_hat, b_hat, figs)
    plot_generator_spectrum(Q_hat, figs / "spectrum_Q_hat.png")

    theorem_obj_val = tv_growth_objective(Q_hat, lambda_ref, cfg.tau_obj)

    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------
    summary = {
        "optimizer_mode": cfg.optimizer_mode,
        "true_parameters": {
            "r_true": cfg.r_true,
            "d_true": cfg.d_true,
            "a_true": cfg.a_true,
            "kappa_left_true": cfg.kappa_left_true,
            "kappa_right_true": cfg.kappa_right_true,
        },
        "accuracy_reference_fit": {
            "theta_accuracy": {
                "r": float(theta_acc[0]),
                "d": float(theta_acc[1]),
                "a": float(theta_acc[2]),
                "kappa_left": float(theta_acc[3]),
                "kappa_right": float(theta_acc[4]),
            },
            "avg_loglik_accuracy": float(avg_loglik_acc),
            "success": bool(res_acc.success),
            "message": str(res_acc.message),
        },
        "estimated_parameters": {
            "r_hat": float(theta_hat[0]),
            "d_hat": float(theta_hat[1]),
            "a_hat": float(theta_hat[2]),
            "kappa_left_hat": float(theta_hat[3]),
            "kappa_right_hat": float(theta_hat[4]),
        },
        "matrix_recovery": {
            "full_generator_relative_error": float(rel_err),
        },
        "likelihoods": {
            "avg_loglik_true": float(ll_true / len(data)),
            "avg_loglik_accuracy": float(ll_acc / len(data)),
            "avg_loglik_fit": float(ll_hat / len(data)),
            "target_avg_loglik_constraint": float(target_avg_loglik),
            "constraint_satisfied": bool((ll_hat / len(data)) >= target_avg_loglik - 1e-8),
        },
        "theorem_flavored_objective": {
            "tau_obj": cfg.tau_obj,
            "likelihood_margin": cfg.likelihood_margin,
            "objective_value": float(theorem_obj_val),
        },
        "dirac_reconstruction": {
            "true_full_relative_reconstruction_error": float(true_rec_err),
            "fit_full_relative_reconstruction_error": float(fit_rec_err),
        },
        "optimization": {
            "success": bool(opt_res.success),
            "message": str(opt_res.message),
            "objective_value": float(opt_res.fun),
            "iterations": int(opt_res.nit) if hasattr(opt_res, "nit") else None,
        },
        "diagnostic_outputs": [
            "kernel_t_*.png",
            "tv_growth.png",
            "lambda_evolution.png",
            "dirac_a.png",
            "dirac_b.png",
            "spectrum_Q_hat.png"
        ],
        "notes": [
            "This script includes the five requested diagnostic outputs.",
            "Start with optimizer_mode='accuracy' for the most stable estimation.",
            "Then compare with theorem_parametric mode."
        ]
    }

    with open(outdir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    report = f"""
Full logistic-Dirac pipeline finished.

Output directory:
  {outdir.resolve()}

Optimizer mode:
  {cfg.optimizer_mode}

True parameters:
  r_true            = {cfg.r_true:.6f}
  d_true            = {cfg.d_true:.6f}
  a_true            = {cfg.a_true:.6f}
  kappa_left_true   = {cfg.kappa_left_true:.6f}
  kappa_right_true  = {cfg.kappa_right_true:.6f}

Estimated parameters:
  r_hat             = {theta_hat[0]:.6f}
  d_hat             = {theta_hat[1]:.6f}
  a_hat             = {theta_hat[2]:.6f}
  kappa_left_hat    = {theta_hat[3]:.6f}
  kappa_right_hat   = {theta_hat[4]:.6f}

Full-generator relative error:
  ||Q_hat - Q_true||_F / ||Q_true||_F = {rel_err:.6e}

Average log-likelihoods:
  true      = {ll_true / len(data):.6f}
  accuracy  = {ll_acc / len(data):.6f}
  fit       = {ll_hat / len(data):.6f}
  target >=   {target_avg_loglik:.6f}

Theorem-like objective:
  (1/tau) log ||exp(tau Q_hat) lambda||_TV = {theorem_obj_val:.6f}
""".strip()

    with open(outdir / "README.txt", "w", encoding="utf-8") as f:
        f.write(report + "\n")

    print(report)
    return summary


if __name__ == "__main__":
    cfg = Config()
    main(cfg)

Full logistic-Dirac pipeline finished.

Output directory:
  /drive/notebooks/full_logistic_dirac_outputs_lightweight_diagnostics

Optimizer mode:
  accuracy

True parameters:
  r_true            = 0.900000
  d_true            = 0.250000
  a_true            = 0.100000
  kappa_left_true   = 2.000000
  kappa_right_true  = 2.000000

Estimated parameters:
  r_hat             = 0.763732
  d_hat             = 0.266132
  a_hat             = 0.122092
  kappa_left_hat    = 1.863749
  kappa_right_hat   = 2.245934

Full-generator relative error:
  ||Q_hat - Q_true||_F / ||Q_true||_F = 6.407937e-02

Average log-likelihoods:
  true      = -1.422505
  accuracy  = -1.421127
  fit       = -1.421127
  target >=   -1.421127

Theorem-like objective:
  (1/tau) log ||exp(tau Q_hat) lambda||_TV = -0.227582
